In [ ]:
from pyspark.sql import SparkSession

# --- Konfigurasjon for Sjekken ---
CASSANDRA_HOST = "127.0.0.1" # Standard loopback
CASSANDRA_PORT = "9042"
# CASSANDRA_USER = "cassandra" # <-- KOMMENTERES UT!
# CASSANDRA_PASS = "cassandra" # <-- KOMMENTERES UT!
CASSANDRA_PACKAGE = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"
# ---

# 1. Initialiser Spark med Cassandra-konfigurasjonene
try:
    spark = (
        SparkSession.builder
        .appName("Cassandra Connection Test")
        
        # Sørger for at pakken blir lastet (Viktig!)
        .config("spark.jars.packages", CASSANDRA_PACKAGE)
        
        # Cassandra tilkoblingsdetaljer
        .config("spark.cassandra.connection.host", CASSANDRA_HOST)
        .config("spark.cassandra.connection.port", CASSANDRA_PORT)
        
        # 🛑 VIKTIG FIKS: Disse er kommentert ut fordi Homebrew Cassandra har auth deaktivert
        # .config("spark.cassandra.auth.username", CASSANDRA_USER) 
        # .config("spark.cassandra.auth.password", CASSANDRA_PASS)
        
        # VIKTIG FOR Å UNNGÅ NETTVERKSFEIL I PYSPARK PÅ LOKAL MASKIN
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        
        .getOrCreate()
    )
    print("✅ Steg 1: Spark Session initialisert.")

    # Sjekk pakke lasting
    print(f"   Konfigurert pakke: {spark.sparkContext.getConf().get('spark.jars.packages')}")
    
except Exception as e:
    print(f"❌ Feil under initialisering av Spark: {e}")
    # Hvis Spark feiler på initiering, prøv å sette CASSANDRA_HOST til "localhost"
    print("\n💡 Forslag: Hvis du får 'Connection refused' her, prøv å endre CASSANDRA_HOST til 'localhost' i filen.")
    exit()

# 2. Forsøk å koble til og lese fra Cassandra
# Bruk en standard systemtabell (keyspace='system', table='local') for en pålitelig sjekk
try:
    print("\nTester tilkobling ved å lese 'system.local'...")
    
    df_check = (
        spark.read
        .format("org.apache.spark.sql.cassandra")
        .options(table="local", keyspace="system")
        .load()
    )
    
    # Prøv å utføre en handling (f.eks. å telle rader)
    count = df_check.count()

    print(f"✅ Steg 2: Tilkobling og lesing lyktes!")
    print(f"   Fant {count} rad(er) i system.local. Cassandra er tilgjengelig og koblet til.")

except Exception as e:
    print("\n❌ Steg 2: Tilkobling eller autentisering feilet.")
    print("   Dette kan skyldes feil host/port, feil brukernavn/passord, eller at Cassandra-noden ikke kjører.")
    print(f"   Detaljert feilmelding: {e}")

finally:
    # Stopper kun hvis Spark ble initialisert (forhindrer krasj på grunn av feil i steg 1)
    if 'spark' in locals():
        spark.stop()
        print("\nSpark Session stoppet.")

In [ ]:
import requests
import pandas as pd
from datetime import datetime
from tqdm import tqdm

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

import pymongo
import json
import os


In [ ]:
spark = (
    SparkSession.builder
    .appName("ElhubConsumptionLoader")
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)


In [ ]:
import requests
import pandas as pd
from tqdm.auto import tqdm
from datetime import date, timedelta

# --- Konstanter ---
API_URL = "https://api.elhub.no/energy-data/v0/price-areas"
DATASET = "CONSUMPTION_PER_GROUP_MBA_HOUR"
AREAS = ["NO1", "NO2", "NO3", "NO4", "NO5"]
YEARS = [2021, 2022, 2023, 2024]


# Hjelper: finn første og siste dato i måneden (yyyy-MM-dd)
def month_date_range(year: int, month: int):
    start = date(year, month, 1)
    if month == 12:
        end = date(year + 1, 1, 1) - timedelta(days=1)
    else:
        end = date(year, month + 1, 1) - timedelta(days=1)
    return start.isoformat(), end.isoformat()


# Hent én måned for ett område
def fetch_consumption_month(area: str, year: int, month: int) -> pd.DataFrame:
    start, end = month_date_range(year, month)

    params = {
        "dataset": DATASET,
        "priceArea": area,
        "startDate": start,   # <-- bare dato, uten T...Z
        "endDate": end,       # <-- bare dato, uten T...Z
    }

    r = requests.get(API_URL, params=params)
    try:
        r.raise_for_status()
    except Exception:
        print(f"❌ API ERROR for {area} {year}-{month:02d}")
        print("URL:", r.url)
        print("Body:", r.text)
        # Returner tom df i stedet for å kaste unntak, så loopen kan fortsette
        return pd.DataFrame()

    data = r.json()
    if not data or "data" not in data:
        return pd.DataFrame()

    df = pd.DataFrame(data["data"])
    if df.empty:
        return df

    df["priceArea"] = area

    # Normaliser tidskolonner (i tilfelle feltnavn varierer)
    if "startTime" in df.columns:
        df["startTime"] = pd.to_datetime(df["startTime"])
    elif "start" in df.columns:
        df.rename(columns={"start": "startTime"}, inplace=True)
        df["startTime"] = pd.to_datetime(df["startTime"])

    if "endTime" in df.columns:
        df["endTime"] = pd.to_datetime(df["endTime"])
    elif "end" in df.columns:
        df.rename(columns={"end": "endTime"}, inplace=True)
        df["endTime"] = pd.to_datetime(df["endTime"])

    return df


# Hent alle måneder for ett prisområde
def fetch_consumption_area(area: str) -> pd.DataFrame:
    frames = []
    for year in YEARS:
        for month in tqdm(range(1, 13), desc=f"{area} {year}"):
            df_m = fetch_consumption_month(area, year, month)
            if not df_m.empty:
                frames.append(df_m)

    if not frames:
        print(f"Ingen data for {area}")
        return pd.DataFrame()

    df_area = pd.concat(frames, ignore_index=True)

    sort_cols = [c for c in ["priceArea", "startTime"] if c in df_area.columns]
    if sort_cols:
        df_area = df_area.sort_values(sort_cols)

    return df_area


# Hent alle områder (NO1–NO5)
def fetch_consumption_all() -> pd.DataFrame:
    all_frames = []
    for area in AREAS:
        print(f"\n=== Fetching {area} ===")
        df_area = fetch_consumption_area(area)
        print(f"{area}: {len(df_area)} rows")
        if not df_area.empty:
            all_frames.append(df_area)

    if not all_frames:
        return pd.DataFrame()

    df_all = pd.concat(all_frames, ignore_index=True)
    sort_cols = [c for c in ["priceArea", "startTime"] if c in df_all.columns]
    if sort_cols:
        df_all = df_all.sort_values(sort_cols)

    return df_all


# Kjør selve hentingen:
df_consumption = fetch_consumption_all()
df_consumption.head()


# TEST

In [2]:
import requests
import pandas as pd
from tqdm.auto import tqdm

API_URL = "https://api.elhub.no/energy-data/v0/price-areas"
DATASET = "CONSUMPTION_PER_GROUP_MBA_HOUR"

AREAS = ["NO1", "NO2", "NO3", "NO4", "NO5"]
YEARS = [2021, 2022, 2023, 2024]


def fetch_month(area: str, year: int, month: int) -> pd.DataFrame:
    """
    Henter én måned for ett område og flater ut 'attributes' til vanlige kolonner.
    Bruker datoformat yyyy-MM-dd som Elhub-API krever.
    """
    # Start = første dag i måneden
    start = f"{year}-{month:02d}-01"

    # End = første dag i neste måned
    if month == 12:
        end = f"{year + 1}-01-01"
    else:
        end = f"{year}-{month + 1:02d}-01"

    params = {
        "dataset": DATASET,
        "priceArea": area,
        "startDate": start,
        "endDate": end,
    }

    r = requests.get(API_URL, params=params, timeout=60)
    try:
        r.raise_for_status()
    except Exception:
        print(f"❌ API ERROR for {area} {year}-{month:02d}:")
        print(r.text)
        raise

    data = r.json()

    if "data" not in data or not data["data"]:
        return pd.DataFrame()

    raw = pd.DataFrame(data["data"])
    if raw.empty:
        return raw

    # 'attributes' er et felt med dict → flates ut til kolonner
    attrs = pd.json_normalize(raw["attributes"])

    # Legg på priceArea eksplisitt (selv om det ofte også ligger inne i attributes)
    attrs["priceArea"] = area

    # Parse tid til datetime hvis de finnes
    for col in ["startTime", "endTime"]:
        if col in attrs.columns:
            attrs[col] = pd.to_datetime(attrs[col])

    # Gi mengde-kolonnen et vettugt navn
    if "quantity" in attrs.columns and "quantityKwh" not in attrs.columns:
        attrs = attrs.rename(columns={"quantity": "quantityKwh"})

    # Elhub bruker typisk 'consumptionGroup' for gruppenavn
    # (hvis kolonnen heter noe annet hos deg, sjekk attrs.columns og tilpass)
    return attrs


def fetch_consumption_area(area: str) -> pd.DataFrame:
    """
    Henter 2021–2024 for ett prisområde og returnerer flatet df.
    """
    frames = []

    for year in YEARS:
        for month in tqdm(range(1, 13), desc=f"{area} {year}"):
            df_m = fetch_month(area, year, month)
            if not df_m.empty:
                frames.append(df_m)

    if not frames:
        return pd.DataFrame()

    df_area = pd.concat(frames, ignore_index=True)

    # Sorter kronologisk per område
    if "startTime" in df_area.columns:
        df_area = df_area.sort_values(["priceArea", "startTime"])

    return df_area


def fetch_consumption_all() -> pd.DataFrame:
    """
    Henter alle områder og alle år, og limer dem sammen.
    """
    all_frames = []

    for area in AREAS:
        print(f"\n=== Fetching {area} ===")
        df_area = fetch_consumption_area(area)
        print(f"{area}: {len(df_area)} rows")
        if not df_area.empty:
            all_frames.append(df_area)

    if not all_frames:
        raise RuntimeError("No data retrieved from Elhub API")

    df_all = pd.concat(all_frames, ignore_index=True)

    if "startTime" in df_all.columns:
        df_all = df_all.sort_values(["priceArea", "startTime"])

    return df_all


# Kjør selve hentingen og lagre til Parquet (brukes av Spark)
df_consumption = fetch_consumption_all()
print("Total rows:", len(df_consumption))
print(df_consumption.head())

df_consumption.to_parquet("consumption_per_group_mba_hour_2021_2024.parquet")
print("✅ Saved to consumption_per_group_mba_hour_2021_2024.parquet")



=== Fetching NO1 ===


NO1 2021:   0%|          | 0/12 [00:00<?, ?it/s]

NO1 2022:   0%|          | 0/12 [00:00<?, ?it/s]

NO1 2023:   0%|          | 0/12 [00:00<?, ?it/s]

NO1 2024:   0%|          | 0/12 [00:00<?, ?it/s]

...

NO5 2021:   0%|          | 0/12 [00:00<?, ?it/s]

NO5 2022:   0%|          | 0/12 [00:00<?, ?it/s]

NO5 2023:   0%|          | 0/12 [00:00<?, ?it/s]

NO5 2024:   0%|          | 0/12 [00:00<?, ?it/s]

NO5: 288 rows
Total rows: 1440
                          consumptionPerGroupMbaHour country  \
0                                                 []      NO   
1  [{'consumptionGroup': 'cabin', 'endTime': '202...      NO   
2  [{'consumptionGroup': 'cabin', 'endTime': '202...      NO   
3  [{'consumptionGroup': 'cabin', 'endTime': '202...      NO   
4  [{'consumptionGroup': 'cabin', 'endTime': '202...      NO   

                eic name priceArea  
0                 *    *       NO1  
1  10YNO-1--------2  NO1       NO1  
2  10YNO-2--------T  NO2       NO1  
3  10YNO-3--------J  NO3       NO1  
4  10YNO-4--------9  NO4       NO1  
✅ Saved to consumption_per_group_mba_hour_2021_2024.parquet


In [2]:
from pyspark.sql import SparkSession, functions as F

# Sett denne hvis du allerede har den definert et annet sted:
MONGODB_URI = "your-mongodb-uri-here"

# 1. Start Spark med Cassandra- og Mongo-connector
spark = (
    SparkSession.builder
    .appName("ElhubConsumptionToCassandraMongo")
    .master("local[*]")
    .config("spark.driver.memory", "4g")  # litt ekstra minne
    .config("spark.jars.packages",
            "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0,"
            "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0")
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.mongodb.write.connection.uri", MONGODB_URI)
    .config("spark.mongodb.write.database", "elhub")
    .getOrCreate()
)

spark

# 2. Les Parquet-filen vi akkurat lagret
cons_raw = spark.read.parquet("consumption_per_group_mba_hour_2021_2024.parquet")
cons_raw.printSchema()
cons_raw.show(5, truncate=False)


root
 |-- consumptionPerGroupMbaHour: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- consumptionGroup: string (nullable = true)
 |    |    |-- endTime: string (nullable = true)
 |    |    |-- lastUpdatedTime: string (nullable = true)
 |    |    |-- meteringPointCount: long (nullable = true)
 |    |    |-- priceArea: string (nullable = true)
 |    |    |-- quantityKwh: double (nullable = true)
 |    |    |-- startTime: string (nullable = true)
 |-- country: string (nullable = true)
 |-- eic: string (nullable = true)
 |-- name: string (nullable = true)
 |-- priceArea: string (nullable = true)



25/11/14 21:12:47 ERROR Executor: Exception in task 2.0 in stage 2.0 (TID 4)/ 4]
org.apache.spark.SparkException: Encountered error while reading file file:///Users/a.h.sheikh/Desktop/IND320_Git_Job/IND320_1_Project/Ass4_Rapporter/consumption_per_group_mba_hour_2021_2024.parquet. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:863)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:293)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.Buffer

ConnectionRefusedError: [Errno 61] Connection refused

ConnectionRefusedError: [Errno 61] Connection refused

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/Users/a.h.sheikh/.pyenv/versions/ind320/lib/python3.12/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/a.h.sheikh/.pyenv/versions/ind320/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/a.h.sheikh/.pyenv/versions/ind320/lib/python3.12/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


# Chat try

In [3]:
import requests
import pandas as pd
from tqdm.auto import tqdm

API_URL = "https://api.elhub.no/energy-data/v0/price-areas"
DATASET = "CONSUMPTION_PER_GROUP_MBA_HOUR"

AREAS = ["NO1", "NO2", "NO3", "NO4", "NO5"]
YEARS = [2021, 2022, 2023, 2024]


def fetch_month(area: str, year: int, month: int) -> pd.DataFrame:
    """
    Henter én måned for ett prisområde og FLATER ut consumptionPerGroupMbaHour.
    Datoformat: yyyy-MM-dd (som Elhub-dokumentasjonen krever).
    """
    # 1. Bygg start og end (første dag i neste måned)
    start = f"{year}-{month:02d}-01"
    if month == 12:
        end = f"{year + 1}-01-01"
    else:
        end = f"{year}-{month + 1:02d}-01"

    params = {
        "dataset": DATASET,
        "priceArea": area,
        "startDate": start,
        "endDate": end,
    }

    r = requests.get(API_URL, params=params, timeout=60)
    try:
        r.raise_for_status()
    except Exception:
        print(f"❌ API ERROR for {area} {year}-{month:02d}:")
        print(r.text)
        raise

    data = r.json()
    if "data" not in data or not data["data"]:
        return pd.DataFrame()

    # Rådf: kolonner: country, eic, name, priceArea, consumptionPerGroupMbaHour
    raw = pd.DataFrame(data["data"])
    if raw.empty:
        return raw

    # 2. Explode array-kolonnen consumptionPerGroupMbaHour
    raw = raw.explode("consumptionPerGroupMbaHour", ignore_index=True)

    # 3. Flate ut structen inni consumptionPerGroupMbaHour til egne kolonner
    nested = pd.json_normalize(raw["consumptionPerGroupMbaHour"])

    # nested har typisk:
    # consumptionGroup, endTime, lastUpdatedTime, meteringPointCount,
    # priceArea, quantityKwh, startTime

    # 4. Slå sammen meta + nested
    df = pd.concat(
        [raw.drop(columns=["consumptionPerGroupMbaHour"]), nested],
        axis=1,
    )

    # 5. Parse tid til datetime
    for col in ["startTime", "endTime", "lastUpdatedTime"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

    # 6. Sørg for at priceArea alltid er satt (bruk nested om nødvendig)
    if "priceArea_x" in df.columns and "priceArea_y" in df.columns:
        df["priceArea"] = df["priceArea_y"].fillna(df["priceArea_x"])
        df = df.drop(columns=["priceArea_x", "priceArea_y"])

    return df


def fetch_consumption_area(area: str) -> pd.DataFrame:
    """Henter 2021–2024 for ett prisområde og limer månedene sammen."""
    frames = []

    for year in YEARS:
        for month in tqdm(range(1, 13), desc=f"{area} {year}"):
            df_m = fetch_month(area, year, month)
            if not df_m.empty:
                frames.append(df_m)

    if not frames:
        return pd.DataFrame()

    df_area = pd.concat(frames, ignore_index=True)

    if "startTime" in df_area.columns:
        df_area = df_area.sort_values(["priceArea", "startTime"])

    return df_area


def fetch_consumption_all() -> pd.DataFrame:
    """Henter alle områder og alle år, og limer alt sammen."""
    all_frames = []

    for area in AREAS:
        print(f"\n=== Fetching {area} ===")
        df_area = fetch_consumption_area(area)
        print(f"{area}: {len(df_area)} rows")
        if not df_area.empty:
            all_frames.append(df_area)

    if not all_frames:
        raise RuntimeError("No data retrieved from Elhub API")

    df_all = pd.concat(all_frames, ignore_index=True)

    if "startTime" in df_all.columns:
        df_all = df_all.sort_values(["priceArea", "startTime"])

    return df_all


# ------------ Kjør henting og lagre til Parquet -----------------
df_consumption = fetch_consumption_all()
print("Total rows:", len(df_consumption))
display(df_consumption.head())

# Valgfritt: behold bare kolonnene du faktisk trenger videre
cols = [
    "priceArea",
    "startTime",
    "endTime",
    "consumptionGroup",
    "quantityKwh",
    "meteringPointCount",
    "country",
    "eic",
    "name",
]
df_consumption = df_consumption[cols]

# Lagre flate data til Parquet – legg den f.eks. i Ass4_Rapporter/
out_path = "Ass4_Rapporter/consumption_per_group_mba_hour_2021_2024.parquet"
df_consumption.to_parquet(out_path)
print("✅ Saved to", out_path)



=== Fetching NO1 ===


NO1 2021:   0%|          | 0/12 [00:00<?, ?it/s]

ERROR! Session/line number was not unique in database. History logging moved to new session 198


KeyError: 'consumptionPerGroupMbaHour'

In [ ]:
pd.read_parquet("Ass4_Rapporter/consumption_per_group_mba_hour_2021_2024.parquet").head()


In [ ]:
from pyspark.sql import SparkSession, functions as F

MONGODB_URI = "DIN_MONGODB_URI_HER"  # bruk samme som du bruker ellers i prosjektet

spark = (
    SparkSession.builder
    .appName("ElhubConsumptionToCassandraMongo")
    .master("local[*]")
    .config("spark.driver.memory", "4g")  # litt ekstra margin
    .config(
        "spark.jars.packages",
        "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0,"
        "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0",
    )
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .config("spark.mongodb.write.connection.uri", MONGODB_URI)
    .config("spark.mongodb.write.database", "elhub")
    .getOrCreate()
)

In [ ]:
cons_raw = spark.read.parquet("Ass4_Rapporter/consumption_per_group_mba_hour_2021_2024.parquet")

cons_raw.printSchema()
cons_raw.limit(5).show(truncate=False)

In [ ]:
# 3. Rens og mapp kolonnenavn til snake_case / lower-case for Cassandra
cons_for_cassandra = cons_raw.select(
    F.col("priceArea").alias("pricearea"),
    F.col("startTime").alias("starttime"),
    F.col("endTime").alias("endtime"),
    F.col("consumptionGroup").alias("consumptiongroup"),
    F.col("quantityKwh").alias("quantitykwh"),
    F.col("meteringPointCount").alias("meteringpointcount"),
)

cons_for_cassandra.printSchema()
cons_for_cassandra.limit(5).show(truncate=False)

In [ ]:
(
    cons_for_cassandra.write
    .format("org.apache.spark.sql.cassandra")
    .mode("append")  # append så vi kan legge til senere hvis vi vil
    .options(
        table="consumption_hourly_by_group",  # <-- DITT tabellnavn
        keyspace="elhub_data",                # <-- DITT keyspace
    )
    .save()
)

print("✅ Written to Cassandra (elhub_data.consumption_hourly_by_group)")

In [ ]:
cons_from_cassandra = (
    spark.read.format("org.apache.spark.sql.cassandra")
    .options(table="consumption_hourly_by_group", keyspace="elhub_data")
    .load()
)

# Mapp til pene Mongo-felter
cons_for_mongo = cons_from_cassandra.select(
    F.col("pricearea").alias("priceArea"),
    F.col("starttime").alias("startTime"),
    F.col("endtime").alias("endTime"),
    F.col("consumptiongroup").alias("consumptionGroup"),
    F.col("quantitykwh").alias("quantityKwh"),
    F.col("meteringpointcount").alias("meteringPointCount"),
)

cons_for_mongo.limit(5).show(truncate=False)

(
    cons_for_mongo.write
    .format("mongodb")
    .mode("overwrite")  # bruk 'append' hvis du vil legge til i eksisterende collection
    .option("collection", "consumption_2021_2024_by_hour")
    .save()
)

print("✅ Written to MongoDB (elhub.consumption_2021_2024_by_hour)")

spark.stop()